# Notebook 02 — Data Cleaning & Preprocessing
## Ames Housing Dataset

This notebook transforms the raw Ames Housing CSV into clean, encoded, and
scaled train/val/test splits saved to `data/processed/`.

**Workflow position:** Runs after Notebook 01 (EDA). Notebook 03 loads its output.

**Output:** Six CSVs (`X_train`, `X_val`, `X_test`, `y_train`, `y_val`, `y_test`)
+ `feature_metadata.json` in `data/processed/`.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')

# ── Paths ────────────────────────────────────────────────────────────────────
RAW_PATH      = Path("../data/raw/AmesHousing.csv")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

📝 **What's happening here** — All imports and path constants are defined once at the top. `PROCESSED_DIR.mkdir(parents=True, exist_ok=True)` creates the output directory if it does not already exist — safe to re-run.

## 1) Load Raw Data

In [ ]:
# Load CSV, normalise column names to CamelCase, drop row-ID columns
df = pd.read_csv(RAW_PATH)
df.columns = df.columns.str.strip().str.replace(' ', '').str.replace('/', '')
df = df.drop(columns=['Order', 'PID'], errors='ignore')
df['SalePrice'] = pd.to_numeric(df['SalePrice'], errors='coerce')

print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols')
print(f'Nulls total: {df.isnull().sum().sum():,}')
df.head(3)

📝 **What's happening here** — The raw CSV header uses spaces (`'Lot Frontage'`). We strip whitespace and join tokens to produce CamelCase names (`'LotFrontage'`) matching the project convention used in `src/config.py` and all downstream code. `Order` and `PID` are index identifiers — not predictors.

## 2) Train / Val / Test Split — 60/20/20 (BEFORE Cleaning)

**Critical rule:** Split first, then clean. If we imputed or scaled the full
dataset before splitting, statistics from val/test rows would leak into training.

In [ ]:
X = df.drop(columns=['SalePrice'])
y = df['SalePrice']

# Step 1: 60% train, 40% holdout
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, random_state=42
)
# Step 2: halve holdout → 20% val + 20% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

total = len(df)
for name, X_s, y_s in [('Train', X_train, y_train),
                         ('Val',   X_val,   y_val),
                         ('Test',  X_test,  y_test)]:
    print(f'{name:<6}: {len(X_s):5d} rows  ({len(X_s)/total*100:.0f}%)'
          f'  | y mean: ${y_s.mean():,.0f}')

📝 **What's happening here** — Two-step splitting: `train_test_split` only produces two subsets, so we call it twice. `random_state=42` makes the split reproducible — every run produces the same rows in each split. The y-mean check quickly confirms all three splits have similar price levels, indicating a balanced random split.

In [ ]:
# Split quality check
split_comparison = pd.DataFrame({
    'Train':      y_train.describe(),
    'Validation': y_val.describe(),
    'Test':       y_test.describe()
}).round(0)
print('SalePrice statistics across splits:')
print(split_comparison)

📝 **What's happening here** — Compare mean, median, and std across all three splits. They should be within ~10% of each other. The median is a more robust comparison than the mean because extreme outliers in one split can inflate or deflate the mean substantially with small split sizes.

## 3) Domain-Driven Imputation Strategy

Nulls in this dataset have two meanings:
1. **Structural absence** — the feature literally does not exist (no pool, no garage)
2. **Measurement gap** — the feature exists but was not recorded

We assign every null column to one of four fill strategies based on this distinction.

In [ ]:
# ── Strategy definitions ────────────────────────────────────────────────────

# Categorical structural absences → fill with 'None'
categorical_none_cols = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1',
    'BsmtFinType2', 'MasVnrType'
]

# Numeric structural absences → fill with 0 (area/count = 0 for absent feature)
numeric_zero_cols = [
    'GarageYrBlt', 'GarageArea', 'GarageCars', 'BsmtFinSF1',
    'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath',
    'BsmtHalfBath', 'MasVnrArea'
]

# Numeric measurement gaps → fill with median (fit on train only)
numeric_median_cols = ['LotFrontage']

# Categorical recording gaps → fill with mode (fit on train only)
categorical_mode_cols = [
    'Electrical', 'MSZoning', 'Utilities', 'Functional',
    'KitchenQual', 'Exterior1st', 'Exterior2nd', 'SaleType'
]

# Sanity check: every null column is handled
null_summary = X_train.isnull().sum()
null_cols = null_summary[null_summary > 0].index.tolist()
all_handled = (categorical_none_cols + numeric_zero_cols
               + numeric_median_cols + categorical_mode_cols)
unhandled = [c for c in null_cols if c not in all_handled]
print(f'Null columns in train: {len(null_cols)}')
print(f'Unhandled: {unhandled if unhandled else "None — all covered ✓"}')

📝 **What's happening here** — Four buckets, determined by the *meaning* of the null — not the dtype alone. `GarageYrBlt` is numeric but its null means 'no garage', so fill with `0` (not the median year). `LotFrontage` is numeric and its null means 'not recorded', so fill with the median of observed houses. The sanity check at the end catches any null column accidentally left out of the strategy lists.

In [ ]:
# Strategy summary table
strategy_rows = []
for col in categorical_none_cols:
    strategy_rows.append({'column': col, 'strategy': "fill 'None'",
                          'reason': 'structural — feature absent'})
for col in numeric_zero_cols:
    strategy_rows.append({'column': col, 'strategy': 'fill 0',
                          'reason': 'structural — area/count = 0'})
for col in numeric_median_cols:
    strategy_rows.append({'column': col, 'strategy': 'median (train only)',
                          'reason': 'unknown — measurement gap'})
for col in categorical_mode_cols:
    strategy_rows.append({'column': col, 'strategy': 'mode (train only)',
                          'reason': 'unknown — recording gap'})

strategy_df = pd.DataFrame(strategy_rows)
print(f'Total columns in imputation log: {len(strategy_df)}')
strategy_df

📝 **What's happening here** — This table is the imputation decision log — a living document that records *why* each column was filled the way it was. When a colleague asks 'why did you fill `BsmtExposure` with `None`?' the answer is in this table: structural absence. Every choice is auditable.

## 4) Apply Imputation (Fit on Train, Transform All Splits)

**Rule:** `SimpleImputer.fit()` runs on `X_train` **only**. The learned statistic
(median / mode) is then applied identically to val and test via `.transform()`.

In [ ]:
def apply_imputation(X_tr, X_v, X_te,
                     cat_none_cols, num_zero_cols,
                     num_median_cols, cat_mode_cols):
    X_tr = X_tr.copy()
    X_v  = X_v.copy()
    X_te = X_te.copy()

    # 1. Categorical structural → 'None'
    for col in cat_none_cols:
        if col in X_tr.columns:
            for split in [X_tr, X_v, X_te]:
                split[col] = split[col].fillna('None')

    # 2. Numeric structural → 0
    for col in num_zero_cols:
        if col in X_tr.columns:
            for split in [X_tr, X_v, X_te]:
                split[col] = split[col].fillna(0)

    # 3. Numeric unknown → median (fit on train only)
    med_imp = SimpleImputer(strategy='median')
    for col in num_median_cols:
        if col in X_tr.columns:
            X_tr[[col]] = med_imp.fit_transform(X_tr[[col]])
            X_v[[col]]  = med_imp.transform(X_v[[col]])
            X_te[[col]] = med_imp.transform(X_te[[col]])

    # 4. Categorical unknown → mode (fit on train only)
    mode_imp = SimpleImputer(strategy='most_frequent')
    for col in cat_mode_cols:
        if col in X_tr.columns:
            X_tr[[col]] = mode_imp.fit_transform(X_tr[[col]])
            X_v[[col]]  = mode_imp.transform(X_v[[col]])
            X_te[[col]] = mode_imp.transform(X_te[[col]])

    return X_tr, X_v, X_te


X_train_imp, X_val_imp, X_test_imp = apply_imputation(
    X_train, X_val, X_test,
    categorical_none_cols, numeric_zero_cols,
    numeric_median_cols, categorical_mode_cols
)
print('Imputation applied.')

📝 **What's happening here** — `apply_imputation` encapsulates the entire imputation workflow. All `.fit()` calls happen on `X_train` data only. `.transform()` applies those same statistics to val and test — they never influence what fill value gets used. Returning imputed copies (not modifying in-place) means we can always compare before/after.

## 5) Missingness Indicators

For `LotFrontage` — the one genuinely unknown numeric null with ~15% missingness —
we add a binary flag **before** imputation clears the null.

In [ ]:
# Add binary flag: 1 = was null (imputed), 0 = observed
for X_orig, X_imp in [(X_train, X_train_imp),
                       (X_val,   X_val_imp),
                       (X_test,  X_test_imp)]:
    X_imp['LotFrontage_was_null'] = X_orig['LotFrontage'].isnull().astype(int)

print('LotFrontage_was_null value counts (train):')
print(X_train_imp['LotFrontage_was_null'].value_counts())

📝 **What's happening here** — We capture the null flag from the *original* un-imputed DataFrame before imputation fills those cells. This binary column lets the model learn whether houses with missing frontage data have systematically different prices — a signal that would be lost if we only kept the imputed numeric value.

In [ ]:
# Correlation of indicator with SalePrice
flag    = X_train_imp['LotFrontage_was_null']
corr    = y_train.corr(flag)

print(f'Correlation of LotFrontage_was_null with SalePrice: {corr:.3f}')
print()
print('SalePrice by missingness:')
comparison = pd.DataFrame({
    'mean':   y_train.groupby(flag).mean().round(0),
    'median': y_train.groupby(flag).median().round(0),
    'count':  y_train.groupby(flag).count()
})
comparison.index = comparison.index.map({0: 'Observed (0)', 1: 'Imputed (1)'})
print(comparison)

📝 **What's happening here** — A weak or zero correlation means the missingness indicator carries little additional signal for the model. A strong correlation (|r| > 0.2) would justify keeping it. Even with weak correlation, the cost of including a binary column is near-zero, so we keep it — tree models will ignore it if it truly adds no value.

## 6) Verify Imputation

In [ ]:
# Check 1: null counts must be zero in all three splits
for name, X_imp in [('Train', X_train_imp),
                     ('Val',   X_val_imp),
                     ('Test',  X_test_imp)]:
    nulls = X_imp.isnull().sum().sum()
    status = '✓' if nulls == 0 else f'✗  {nulls} nulls remain!'
    print(f'{name:<6}: {nulls} nulls  {status}')

assert X_train_imp.isnull().sum().sum() == 0, "Nulls remain after imputation!"

📝 **What's happening here** — This assertion fails immediately if any null column was accidentally left out of the strategy lists. Catching the error here — at the imputation step — is far better than getting a cryptic error deep inside sklearn when fitting the model.

In [ ]:
# Check 2: LotFrontage distribution before vs after (visualise median fill spike)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

X_train['LotFrontage'].dropna().hist(bins=30, ax=axes[0],
                                      color='steelblue', edgecolor='white')
axes[0].set_title(f'LotFrontage — Before '
                  f'(n={X_train["LotFrontage"].notna().sum()} observed)')
axes[0].set_xlabel('feet')

X_train_imp['LotFrontage'].hist(bins=30, ax=axes[1],
                                  color='orange', edgecolor='white')
train_median = X_train['LotFrontage'].median()
axes[1].axvline(train_median, color='red', linestyle='--',
                label=f'Median fill = {train_median:.0f} ft')
axes[1].set_title(f'LotFrontage — After '
                  f'(n={len(X_train_imp)} total)')
axes[1].set_xlabel('feet')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f'Spike at {train_median:.0f} ft = '
      f'{X_train["LotFrontage"].isna().sum()} imputed rows.')

📝 **What's happening here** — The 'before' histogram shows the observed distribution. The 'after' histogram shows a spike at the median fill value — all imputed rows now hold the exact same number. This compresses variance: imputed rows are indistinguishable from each other on `LotFrontage`. For a small fraction (~15%), this is an acceptable trade-off. KNN imputation (shown next) preserves more variance at higher compute cost.

In [ ]:
# Imputation method comparison: median-fill vs KNN for LotFrontage
numeric_only_cols = X_train.select_dtypes(include='number').columns.tolist()
X_tr_num = X_train[numeric_only_cols].copy()
X_val_num = X_val[numeric_only_cols].copy()

# Pre-fill other numeric nulls with median so KNN can use them as neighbors
pre_imp = SimpleImputer(strategy='median')
X_tr_pre  = pd.DataFrame(pre_imp.fit_transform(X_tr_num), columns=numeric_only_cols)
X_val_pre = pd.DataFrame(pre_imp.transform(X_val_num),   columns=numeric_only_cols)

# Restore LotFrontage null so KNN imputes it
X_tr_pre['LotFrontage']  = X_train['LotFrontage'].values
X_val_pre['LotFrontage'] = X_val['LotFrontage'].values

knn_imp = KNNImputer(n_neighbors=5)
X_tr_knn  = pd.DataFrame(knn_imp.fit_transform(X_tr_pre),  columns=numeric_only_cols)

# Compare imputed values for originally-null rows
null_mask = X_train['LotFrontage'].isnull().values
median_vals = X_train_imp.loc[X_train_imp.index[null_mask], 'LotFrontage']
knn_vals    = X_tr_knn['LotFrontage'].values[null_mask]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(median_vals, bins=20, alpha=0.6, label='Median fill', color='steelblue')
axes[0].hist(knn_vals,    bins=20, alpha=0.6, label='KNN fill (k=5)',  color='orange')
axes[0].set_title('LotFrontage — Imputed Values Only')
axes[0].set_xlabel('feet')
axes[0].legend()

axes[1].scatter(median_vals.values, knn_vals, alpha=0.4, s=15)
min_v = min(float(median_vals.min()), knn_vals.min())
max_v = max(float(median_vals.max()), knn_vals.max())
axes[1].plot([min_v, max_v], [min_v, max_v], 'r--', label='y=x')
axes[1].set_xlabel('Median fill')
axes[1].set_ylabel('KNN fill')
axes[1].set_title('Median vs KNN Imputed Values')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f'Median fill: all imputed values = {float(median_vals.iloc[0]):.1f} ft')
print(f'KNN fill:    range {knn_vals.min():.1f} — {knn_vals.max():.1f} ft')

📝 **What's happening here** — KNN imputation estimates each missing value from its 5 nearest neighbours in feature space, producing varied fill values instead of a single spike. This preserves more variance but is O(n²) — slow on large datasets. For `LotFrontage` (~15% null), median imputation is the default choice. Use KNN only when the column is a high-importance predictor and the dataset is small enough to afford the compute cost.

## 7) Outlier Removal

We remove two known `GrLivArea` outliers from the **training set only**.
They are partial-sale properties with unusually low prices for their size.

In [ ]:
before = len(X_train_imp)

outlier_mask = (X_train_imp['GrLivArea'] > 4000) & (y_train < 200_000)
X_train_imp = X_train_imp[~outlier_mask]
y_train      = y_train[~outlier_mask]

print(f'Train rows before: {before}')
print(f'Train rows after:  {len(X_train_imp)} ({before - len(X_train_imp)} removed)')

📝 **What's happening here** — Outliers are removed from training only — val and test simulate production data that we cannot filter. These two houses have very large living areas but sold at below-market prices (likely forced/partial sales). Including them distorts the model's coefficient for `GrLivArea`, the strongest square-footage predictor.

## 8) Feature Engineering

We create four composite features that capture domain relationships more directly
than any single raw column.

In [ ]:
def engineer_features(X):
    X = X.copy()

    # Total square footage: first floor + second floor + basement
    sf_cols = ['1stFlrSF', '2ndFlrSF', 'TotalBsmtSF']
    present_sf = [c for c in sf_cols if c in X.columns]
    X['TotalSF'] = X[present_sf].fillna(0).sum(axis=1)

    # House age at sale
    if 'YrSold' in X.columns and 'YearBuilt' in X.columns:
        X['HouseAge'] = X['YrSold'] - X['YearBuilt']

    # Total bathrooms (full = 1, half = 0.5)
    bath_cols = {'FullBath': 1, 'HalfBath': 0.5,
                 'BsmtFullBath': 1, 'BsmtHalfBath': 0.5}
    total_bath = sum(X[c].fillna(0) * w
                     for c, w in bath_cols.items() if c in X.columns)
    X['TotalBath'] = total_bath

    # Binary: was the house remodelled?
    if 'YearBuilt' in X.columns and 'YearRemodAdd' in X.columns:
        X['HasRemodel'] = (X['YearRemodAdd'] > X['YearBuilt']).astype(int)

    return X


X_train_imp = engineer_features(X_train_imp)
X_val_imp   = engineer_features(X_val_imp)
X_test_imp  = engineer_features(X_test_imp)

new_cols = ['TotalSF', 'HouseAge', 'TotalBath', 'HasRemodel']
existing = [c for c in new_cols if c in X_train_imp.columns]
print(f'Engineered features added: {existing}')
print(X_train_imp[existing].describe().round(1))

📝 **What's happening here** — `TotalSF` aggregates all square footage — often more predictive than individual floor areas because buyers think in total size. `HouseAge` converts the year-built timestamp into an age that the model can reason about linearly. `TotalBath` weights full baths as 1 and half baths as 0.5 (industry convention). `HasRemodel` is a binary flag — renovated houses tend to command a premium regardless of age.

## 9) Categorical Feature Audit — Ordinal vs Nominal

In [ ]:
cat_cols = X_train_imp.select_dtypes(include='object').columns.tolist()
print(f'Total categorical columns remaining: {len(cat_cols)}')

for col in cat_cols:
    print(f'  {col:<20}: {X_train_imp[col].nunique()} unique')

📝 **What's happening here** — Listing all remaining object-dtype columns with their unique value counts confirms that imputation converted all nulls to the `'None'` string (now appearing in value counts) and that no unexpected categories crept in. Cardinality informs the encoder choice in the next two sections.

In [ ]:
# Value count bar charts for a sample of categoricals
sample_cat = cat_cols[:4]
fig, axes = plt.subplots(1, len(sample_cat), figsize=(16, 4))

for ax, col in zip(axes, sample_cat):
    vc = X_train_imp[col].value_counts().head(10)
    vc.plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Value Counts — Sample Categorical Columns (post-imputation)', y=1.02)
plt.tight_layout()
plt.show()

📝 **What's happening here** — The bar charts confirm that `'None'` is now the dominant category in structural-absence columns (e.g. `PoolQC`, `Alley`). This is the correct outcome — the model will see `'None'` as its own valid category, not a missing value, and can learn the 'no pool / no alley' signal.

## 10) Ordinal Encoding

Quality and condition columns have a natural ordering: `Po < Fa < TA < Gd < Ex`.
We use `OrdinalEncoder` with an explicit category list to preserve this rank.

In [ ]:
quality_order = ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex']

ordinal_cols = [
    'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
    'HeatingQC', 'KitchenQual', 'FireplaceQu',
    'GarageQual', 'GarageCond', 'PoolQC'
]
present_ord = [c for c in ordinal_cols if c in X_train_imp.columns]

ordinal_encoder = OrdinalEncoder(
    categories=[quality_order] * len(present_ord),
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

X_train_imp[present_ord] = ordinal_encoder.fit_transform(X_train_imp[present_ord])
X_val_imp[present_ord]   = ordinal_encoder.transform(X_val_imp[present_ord])
X_test_imp[present_ord]  = ordinal_encoder.transform(X_test_imp[present_ord])

print('Ordinal encoding applied.')
print('ExterQual sample values after encoding:')
print(X_train_imp['ExterQual'].value_counts().sort_index())

📝 **What's happening here** — The `categories` parameter explicitly controls the integer mapping — if we accidentally reversed the list, the model would learn that poor quality predicts higher prices. `'None'` maps to 0 (below Poor), which is semantically correct: a house with no basement has lower quality on the basement dimension than one with a poor basement. `handle_unknown='use_encoded_value'` with `-1` safely handles any unseen category at inference time without crashing.

In [ ]:
# Before/after bar chart for ExterQual
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Before: raw string distribution from df
df['ExterQual'].value_counts().reindex(quality_order, fill_value=0).plot(
    kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('ExterQual — Before (string categories)')
axes[0].tick_params(axis='x', rotation=0)

# After: integer distribution in X_train_imp
pd.Series(X_train_imp['ExterQual']).value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color='coral')
axes[1].set_title('ExterQual — After (ordinal integers)')
axes[1].set_xlabel('Encoded value (0=None, 5=Ex)')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

📝 **What's happening here** — The bar shapes are identical — the same proportion of each category — but the x-axis labels changed from strings to integers. The model now receives a single numeric axis for quality, preserving the rank: a house with `ExterQual=5` (Ex) is unambiguously better than one with `ExterQual=3` (TA). OHE would have created 6 independent columns, losing this monotonic structure.

## 11) One-Hot Encoding

Nominal columns (no inherent rank) are encoded as binary indicator columns.
We use `handle_unknown='ignore'` so unseen categories at inference time
are encoded as all-zeros rather than crashing the pipeline.

In [ ]:
nominal_cols = [c for c in X_train_imp.select_dtypes(include='object').columns]
print(f'Nominal columns to OHE: {len(nominal_cols)}')

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe_train = ohe.fit_transform(X_train_imp[nominal_cols])
ohe_val   = ohe.transform(X_val_imp[nominal_cols])
ohe_test  = ohe.transform(X_test_imp[nominal_cols])

ohe_feature_names = ohe.get_feature_names_out(nominal_cols)

X_train_enc = pd.concat([
    X_train_imp.drop(columns=nominal_cols),
    pd.DataFrame(ohe_train, columns=ohe_feature_names, index=X_train_imp.index)
], axis=1)
X_val_enc = pd.concat([
    X_val_imp.drop(columns=nominal_cols),
    pd.DataFrame(ohe_val,   columns=ohe_feature_names, index=X_val_imp.index)
], axis=1)
X_test_enc = pd.concat([
    X_test_imp.drop(columns=nominal_cols),
    pd.DataFrame(ohe_test,  columns=ohe_feature_names, index=X_test_imp.index)
], axis=1)

print(f'Shapes after OHE:')
print(f'  train={X_train_enc.shape}, val={X_val_enc.shape}, test={X_test_enc.shape}')
assert X_train_enc.shape[1] == X_val_enc.shape[1] == X_test_enc.shape[1], \
    'OHE column mismatch!'
print('Column count consistent across splits ✓')

📝 **What's happening here** — OHE expands each nominal column into N binary columns (one per unique category). 33 nominal columns → ~200+ binary columns, a 6× expansion. The assertion confirms all three splits have the same column count — if `handle_unknown='error'` were set, unseen val/test categories would silently produce a different column count and break all downstream code.

In [ ]:
# Sparsity check + high cardinality audit
ohe_df = pd.DataFrame(ohe_train, columns=ohe_feature_names)
sparsity = ohe_df.mean().sort_values(ascending=False)

print('Top 10 most common OHE features (% rows = 1):')
print((sparsity.head(10) * 100).round(1).to_string())

print('\nBottom 10 rarest OHE features (% rows = 1):')
print((sparsity.tail(10) * 100).round(1).to_string())

# High cardinality audit
cardinality = {c: X_train_imp[c].nunique()
               for c in nominal_cols}
high_card = {c: n for c, n in cardinality.items() if n > 15}
print(f'\nHigh-cardinality columns (>15 unique): {high_card}')

📝 **What's happening here** — Features that are 1 for fewer than 0.5% of rows carry almost no signal — the model has too few examples to learn a reliable coefficient. These near-constant OHE columns are noise candidates for Week 3 feature pruning. High-cardinality columns like `Neighborhood` (25 unique) produce many OHE columns; target encoding would be more efficient but is deferred to Week 3 as a refinement.

## 12) Numeric Scaling

`StandardScaler` subtracts the training mean and divides by the training std,
producing features with mean ≈ 0 and std = 1. Fit on train only.

In [ ]:
numeric_scale_cols = X_train_enc.select_dtypes(include=[np.number]).columns.tolist()
print(f'Numeric columns to scale: {len(numeric_scale_cols)}')

scaler = StandardScaler()
X_train_enc[numeric_scale_cols] = scaler.fit_transform(X_train_enc[numeric_scale_cols])
X_val_enc[numeric_scale_cols]   = scaler.transform(X_val_enc[numeric_scale_cols])
X_test_enc[numeric_scale_cols]  = scaler.transform(X_test_enc[numeric_scale_cols])

print('StandardScaler applied.')
print('Post-scaling stats for 5 columns:')
print(X_train_enc[numeric_scale_cols[:5]].describe().round(3).loc[['mean', 'std']])

📝 **What's happening here** — All numeric columns — including the OHE binary columns — are passed through the scaler. For OHE columns this is technically unnecessary (they are already 0/1), but a uniform scaler avoids conditional logic and keeps the preprocessing code simple. After scaling, `LotArea` (range ~215,000) and `OverallQual` (range 8) live on the same numeric axis — regularised models (Ridge, Lasso) can now compare their coefficients meaningfully.

In [ ]:
# StandardScaler vs RobustScaler on LotArea (has outliers)
from sklearn.preprocessing import RobustScaler

col = 'LotArea'
raw_vals = df[col].dropna()

std_sc = StandardScaler()
rob_sc = RobustScaler()
std_scaled = std_sc.fit_transform(raw_vals.values.reshape(-1, 1)).flatten()
rob_scaled = rob_sc.fit_transform(raw_vals.values.reshape(-1, 1)).flatten()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(raw_vals, bins=40, color='steelblue')
axes[0].set_title('LotArea (raw)')
axes[1].hist(std_scaled, bins=40, color='coral')
axes[1].set_title('StandardScaler')
axes[2].hist(rob_scaled, bins=40, color='green', alpha=0.7)
axes[2].set_title('RobustScaler')
plt.suptitle('LotArea: Raw vs StandardScaler vs RobustScaler', fontsize=13)
plt.tight_layout()
plt.show()

📝 **What's happening here** — `RobustScaler` uses the IQR instead of std — it is immune to the influence of extreme outliers. For `LotArea`, which has a large outlier (~215,000 sqft), `RobustScaler` preserves more spread in the typical range. We use `StandardScaler` here for simplicity and consistency with the sklearn Pipeline in Notebook 03; `RobustScaler` is a drop-in replacement for future refinements.

## 13) Final Verification

In [ ]:
# Shapes and null counts
for name, X_enc in [('Train', X_train_enc),
                     ('Val',   X_val_enc),
                     ('Test',  X_test_enc)]:
    nulls = X_enc.isnull().sum().sum()
    print(f'{name:<6}: shape={X_enc.shape}  nulls={nulls}')

# Dtype audit — no object columns must remain
dtypes = X_train_enc.dtypes.value_counts()
print('\nX_train_enc dtype counts:')
print(dtypes)
assert 'object' not in dtypes.index, 'ERROR: object columns still present!'
print('\nAll columns numeric — dataset is model-ready ✓')

📝 **What's happening here** — The dtype assertion is the final gate before saving. If any categorical column was accidentally left unencoded (e.g. a column added during feature engineering that was not in the OHE list), this assertion fires here rather than as a cryptic error inside sklearn at training time.

## 14) Save Processed Data

Save six CSV files + `feature_metadata.json` to `data/processed/`.
Notebook 03 loads these directly — no re-processing needed.

In [ ]:
# Save X splits (encoded + scaled) and y splits
X_train_enc.to_csv(PROCESSED_DIR / 'X_train.csv')
X_val_enc.to_csv(PROCESSED_DIR  / 'X_val.csv')
X_test_enc.to_csv(PROCESSED_DIR  / 'X_test.csv')

y_train.to_csv(PROCESSED_DIR / 'y_train.csv', header=True)
y_val.to_csv(PROCESSED_DIR   / 'y_val.csv',   header=True)
y_test.to_csv(PROCESSED_DIR  / 'y_test.csv',  header=True)

# Save feature metadata for Notebook 03
ordinal_orders = {
    col: quality_order for col in present_ord
}

feature_metadata = {
    'numeric_features':  [c for c in numeric_scale_cols
                          if c not in present_ord
                          and not any(c.startswith(n + '_') for n in nominal_cols)],
    'ordinal_features':  present_ord,
    'nominal_features':  nominal_cols,
    'ordinal_orders':    ordinal_orders,
    'selected_features': [
        'OverallQual', 'TotalSF', 'GarageCars', 'TotalBath',
        'YearBuilt', 'TotalBsmtSF', 'KitchenQual', 'BsmtQual',
        'Neighborhood', 'ExterQual'
    ],
}

with open(PROCESSED_DIR / 'feature_metadata.json', 'w') as f:
    json.dump(feature_metadata, f, indent=2)

print('Saved to data/processed/:')
for p in sorted(PROCESSED_DIR.iterdir()):
    size_kb = p.stat().st_size / 1024
    print(f'  {p.name:<30}  {size_kb:6.1f} KB')

📝 **What's happening here** — Six CSV files: three feature matrices (`X_*`) and three target vectors (`y_*`). The index is preserved so rows can be aligned back if needed. `feature_metadata.json` carries the column-group assignments and ordinal orderings that Notebook 03 needs to reconstruct the sklearn `ColumnTransformer` correctly — without it, Notebook 03 would have to re-derive these from scratch.

In [ ]:
# Reload and verify
X_check = pd.read_csv(PROCESSED_DIR / 'X_train.csv', index_col=0)
y_check = pd.read_csv(PROCESSED_DIR / 'y_train.csv', index_col=0)

print(f'Reloaded X_train: {X_check.shape}')
print(f'Reloaded y_train: {y_check.shape}')
print(f'Nulls: {X_check.isnull().sum().sum()}')
print(f'Dtypes unique: {X_check.dtypes.unique()}')
print(f'y_train sample: {y_check.squeeze().head(3).values}')

📝 **What's happening here** — Reload from disk and check shape, null count, and dtypes. This confirms the CSV was written and read correctly — CSV round-trips can sometimes change dtypes (e.g. bool → int64) or introduce index misalignment. Catching problems here is faster than debugging in Notebook 03 after the fact.

## 15) Summary

| Step | Output |
|------|--------|
| Load raw CSV | 2,930 rows × 80 features |
| 60/20/20 split (before cleaning) | Train / Val / Test — no leakage |
| Domain-driven imputation | All splits null-free |
| Missingness indicator | `LotFrontage_was_null` added |
| 2 outliers removed | From training only |
| Feature engineering | `TotalSF`, `HouseAge`, `TotalBath`, `HasRemodel` |
| Ordinal encoding | 10 quality columns → integers 0–5 |
| One-hot encoding | Nominal columns → binary indicators |
| StandardScaler | All numeric columns centred and scaled |
| Save to disk | 6 CSVs + feature_metadata.json → `data/processed/` |

**Next step:** Notebook 03 — Feature Selection & Model Training